In [1]:
import torch
import torch.optim as optim
import torch.nn as nn

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda:0


In [3]:
from torchvision import transforms
from data.dataset import CustomImageDataset

train_transforms = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.RandomGrayscale(p=0.1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CustomImageDataset(root_dir='../dataset/train', transform=train_transforms)
test_dataset  = CustomImageDataset(root_dir='../dataset/test', transform=test_transforms)

In [4]:
from torch.utils.data import WeightedRandomSampler

class_counts = train_dataset.get_class_counts()

weights_per_class = {
    0: 1.0 / class_counts[0],
    1: 1.0 / class_counts[1]
}

sample_weights = [weights_per_class[label] for label in train_dataset.labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [5]:
from classifier.model import ImageArtifactClassifierCNN, ImageArtifactClassifierSimpleCNN, ResNet18ArtifactClassifier, EfficientNetArtifactClassifier
from scripts.helpers import random_search

In [6]:
param_space_cnn = {
    'lr': [1e-3, 5e-4, 1e-4],
    'batch_size': [16, 32, 64],
    'weight_decay': [0.0, 1e-4, 1e-5]
}

print("=== Custom Simple CNN Tuning (Random Search) ===")
best_cnn_simple_params = random_search(
    lambda: ImageArtifactClassifierSimpleCNN(num_classes=2),
    train_dataset,
    test_dataset,
    param_space_cnn,
    sampler,
    num_trials=6, 
    device=device
)

=== Custom Simple CNN Tuning (Random Search) ===
Starting Random Search. Total trials: 6

--- Experiment 1/6 | LR: 0.0001, Batch Size: 64, Weight Decay: 0.0001 ---
Result of combination 1: Micro F1 = 0.6650

--- Experiment 2/6 | LR: 0.0001, Batch Size: 64, Weight Decay: 0.0 ---
Result of combination 2: Micro F1 = 0.7750

--- Experiment 3/6 | LR: 0.001, Batch Size: 16, Weight Decay: 1e-05 ---
Result of combination 3: Micro F1 = 0.6150

--- Experiment 4/6 | LR: 0.0001, Batch Size: 16, Weight Decay: 1e-05 ---
Result of combination 4: Micro F1 = 0.7100

--- Experiment 5/6 | LR: 0.001, Batch Size: 16, Weight Decay: 0.0 ---
Result of combination 5: Micro F1 = 0.7650

--- Experiment 6/6 | LR: 0.0001, Batch Size: 64, Weight Decay: 1e-05 ---
Result of combination 6: Micro F1 = 0.6800

🏆 Best parameters: {'lr': 0.0001, 'batch_size': 64, 'weight_decay': 0.0} (F1: 0.7750)


In [7]:
print("=== Custom Better CNN Tuning (Random Search) ===")
best_cnn_better_params = random_search(
    lambda: ImageArtifactClassifierCNN(num_classes=2),
    train_dataset,
    test_dataset,
    param_space_cnn,
    sampler,
    num_trials=6, 
    device=device
)

=== Custom Better CNN Tuning (Random Search) ===
Starting Random Search. Total trials: 6

--- Experiment 1/6 | LR: 0.0001, Batch Size: 16, Weight Decay: 0.0 ---
Result of combination 1: Micro F1 = 0.7250

--- Experiment 2/6 | LR: 0.001, Batch Size: 64, Weight Decay: 0.0001 ---
Result of combination 2: Micro F1 = 0.8300

--- Experiment 3/6 | LR: 0.0005, Batch Size: 32, Weight Decay: 1e-05 ---
Result of combination 3: Micro F1 = 0.8100

--- Experiment 4/6 | LR: 0.001, Batch Size: 32, Weight Decay: 1e-05 ---
Result of combination 4: Micro F1 = 0.8300

--- Experiment 5/6 | LR: 0.001, Batch Size: 64, Weight Decay: 0.0001 ---


KeyboardInterrupt: 

In [ ]:
param_space_resnet = {
    'lr': [1e-4, 5e-5, 1e-5],
    'batch_size': [16, 32],
    'weight_decay': [0.0, 1e-5]
}

print("\n=== ResNet-18 Tuning (Random Search) ===")
best_resnet_params = random_search(
    lambda: ResNet18ArtifactClassifier(num_classes=2, pretrained=True),
    train_dataset,
    test_dataset,
    param_space_resnet,
    sampler,
    num_trials=4,
    device=device
)

In [ ]:
param_space_efficient = {
    'lr': [1e-4, 5e-5, 1e-5], 
    'batch_size': [16, 32], 
    'weight_decay': [1e-4, 1e-5] 
}

print("\n=== EfficientNet-B0 Tuning ===")
best_eff_params = random_search(
    lambda: EfficientNetArtifactClassifier(num_classes=2, pretrained=True),
    train_dataset,
    test_dataset,
    param_space_efficient,
    sampler,
    num_trials=4,
    device=device
)

In [ ]:
from torch.utils.data import DataLoader

train_loader_simple_cnn = DataLoader(
    train_dataset,
    batch_size=best_cnn_simple_params['batch_size'],
    sampler=sampler,
    num_workers=4
)

train_loader_better_cnn = DataLoader(
    train_dataset,
    batch_size=best_cnn_better_params['batch_size'],
    sampler=sampler,
    num_workers=4
)

train_loader_resnet = DataLoader(
    train_dataset,
    batch_size=best_resnet_params['batch_size'],
    sampler=sampler,
    num_workers=4
)

train_loader_efficientnet = DataLoader(
    train_dataset,
    batch_size=best_eff_params['batch_size'],
    sampler=sampler,
    num_workers=4
)

test_loader  = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2)

In [ ]:
from scripts.helpers import show_augmented_images

print("Examples of Augmented Images (Train Set):")

dataiter = iter(train_loader_simple_cnn)
images, labels = next(dataiter)

print(f"image batch dim: {images.shape}")
print(f"label batch dim: {labels.shape}")
print(f"final layer dim: {labels.unique().shape}")
print(f"class 0 count: {labels[labels == 0].shape[0]}")
print(f"class 1 count: {labels[labels == 1].shape[0]}")

show_augmented_images(train_loader_simple_cnn, num_images=8)

In [ ]:
simple_cnn_model = ImageArtifactClassifierSimpleCNN(num_classes=2).to(device)

better_cnn_model = ImageArtifactClassifierCNN(num_classes=2).to(device)

resnet_model = ResNet18ArtifactClassifier(num_classes=2, pretrained=True).to(device)

efficientnet_model = EfficientNetArtifactClassifier(num_classes=2, pretrained=True).to(device)

criterion = nn.CrossEntropyLoss()

optimizer_simple_cnn = optim.Adam(
    simple_cnn_model.parameters(),
    lr=best_cnn_simple_params['lr'],
    weight_decay=best_cnn_simple_params['weight_decay'])

optimizer_better_cnn = optim.Adam(
    better_cnn_model.parameters(),
    lr=best_cnn_better_params['lr'],
    weight_decay=best_cnn_better_params['weight_decay'])

optimizer_efficientnet = optim.Adam(
    efficientnet_model.parameters(),
    lr=best_eff_params['lr'],
    weight_decay=best_eff_params['weight_decay'])

optimizer_resnet = optim.Adam(
    resnet_model.parameters(),
    lr=best_resnet_params['lr'],
    weight_decay=best_resnet_params['weight_decay'])

In [ ]:
from scripts.helpers import train_model_with_metrics

trained_model_simple_cnn, training_history_simple_cnn = train_model_with_metrics(
    simple_cnn_model,
    train_loader_simple_cnn,
    test_loader,
    criterion,
    optimizer_simple_cnn,
    num_epochs=15,
    device=device)

In [ ]:
trained_model_better_cnn, training_history_better_cnn = train_model_with_metrics(
    better_cnn_model,
    train_loader_better_cnn,
    test_loader,
    criterion,
    optimizer_better_cnn,
    num_epochs=15,
    device=device)

In [ ]:
trained_model_resnet, training_history_resnet = train_model_with_metrics(
    resnet_model,
    train_loader_resnet,
    test_loader,
    criterion,
    optimizer_resnet,
    num_epochs=15,
    device=device)

In [ ]:
trained_model_efficientnet, training_history_efficientnet = train_model_with_metrics(
    efficientnet_model,
    train_loader_efficientnet,
    test_loader,
    criterion,
    optimizer_efficientnet,
    num_epochs=15,
    device=device)

In [ ]:
from scripts.helpers import plot_training_history, evaluate_confusion_matrix

plot_training_history(training_history_simple_cnn, title="Simple CNN Training History")
evaluate_confusion_matrix(trained_model_simple_cnn, test_loader, device=device)

In [ ]:
plot_training_history(training_history_better_cnn, title="Better CNN Training History")
evaluate_confusion_matrix(trained_model_better_cnn, test_loader, device=device)

In [ ]:
plot_training_history(training_history_resnet)
evaluate_confusion_matrix(trained_model_resnet, test_loader, device=device)

In [ ]:
plot_training_history(training_history_efficientnet)
evaluate_confusion_matrix(trained_model_efficientnet, test_loader, device=device)

In [ ]:
from scripts.helpers import compare_models

models_to_test = {
    'Simple CNN': trained_model_simple_cnn,
    'Better CNN': trained_model_better_cnn,
    'ResNet-18': trained_model_resnet,
    'EfficientNet-B0': trained_model_efficientnet
}

df_comparison = compare_models(models_to_test, test_loader, device)